# Green alleys pipeline: ruelles vertes to GeoJSON

This notebook adds a second layer to CatMap: Montréal's **ruelles vertes** (green alleys).

```
data/ruelles-vertes.json     input: the City of Montréal's open data, with the alley shapes
data/ruelles_vertes.geojson  output, written by this notebook
site/app.js                  draws it under the cat pins
```

The portal's CSV (`ruelles-vertes.csv`) has the same columns but **no coordinates**, so it can't go on a map. The JSON version has the shapes, so this follows Lab 4's Recipe 3. Run each cell in order, top to bottom.

## 1) Import libraries and load the file
GeoPandas reads the shapes and the table together. We print the coordinate system, how many alleys there are, what kind of shapes they are and the column names.

In [1]:
import os
import pandas as pd
import geopandas as gpd

in_path = "../data/ruelles-vertes.json"
gdf = gpd.read_file(in_path)
print("CRS:", gdf.crs)
print(len(gdf), "alley segments,", gdf["RUELLE_ID"].nunique(), "green alleys")
print("Geometry types:", sorted(gdf.geom_type.unique()))
print(gdf.columns.tolist())

CRS: EPSG:4326
1262 alley segments, 446 green alleys
Geometry types: ['Polygon']
['ID_TRC', 'PROPRIETAIRE_REF', 'CODE_ARR', 'FORME', 'RUELLE_ID', 'DATE_AMENAGEMENT', 'DATE_BONIFICATION', 'CIRCULATION', 'TYPE_AMENAGEMENT', 'TYPE_RECOUVREMENT', 'SUPERFICIE_RUELLE_M2', 'SUPERFICIE_VERDISSEMENT_M2', 'NUMERO_DE_RESOLUTION', 'DATE_MAJ', 'geometry']


## 2) Keep only what the popups show
A GeoJSON file carries every column you leave in, so we keep six and give them short English names. `year` comes from the date the alley was greened; 17 rows have no date and stay blank. Values like `Ouverte` stay in French, as the city wrote them.

No filter: this MVP shows **all of Montréal** (1,262 segments), and the file stays well under 5 MB.

In [2]:
keep = gpd.GeoDataFrame({
    "borough": gdf["PROPRIETAIRE_REF"],
    "alley_id": gdf["RUELLE_ID"],
    "year": pd.to_datetime(gdf["DATE_AMENAGEMENT"], errors="coerce").dt.year.astype("Int64"),
    "circulation": gdf["CIRCULATION"],           # Ouverte / Fermée / Partiellement fermée ...
    "greening": gdf["TYPE_AMENAGEMENT"],         # Plates-bandes, Mobilier ...
    "area_m2": gdf["SUPERFICIE_RUELLE_M2"].round(0),
}, geometry=gdf.geometry, crs=gdf.crs)

print("before:", len(gdf), "after:", len(keep))   # same: no filter
keep.head()

before: 1262 after: 1262


,borough,alley_id,year,circulation,greening,area_m2,geometry
0,Rosemont - La Petite-Patrie,RPP-L-1000048,2014,Ouverte,Plates-bandes et mobilier,180.0,"POLYGON ((-73.605 45.54543, -73.60492 45.54551..."
1,Plateau - Mont-Royal,PMR-H-1000064,2017,Fermée,Plates-bandes,88.0,"POLYGON ((-73.56587 45.53057, -73.56572 45.530..."
2,Plateau - Mont-Royal,PMR-H-1000064,2019,Ouverte,NaN,93.0,"POLYGON ((-73.56408 45.52973, -73.56392 45.529..."
3,Verdun,VRD-H-1020143,2014,Ouverte,Plates-bandes,165.0,"POLYGON ((-73.57637 45.44881, -73.57635 45.449..."
4,Outremont,OUT-I-1000000,2009,Ouverte,Mobilier,310.0,"POLYGON ((-73.60909 45.52184, -73.60889 45.522..."


## 3) Convert to longitude and latitude, then write
The file is already in longitude and latitude (`EPSG:4326`), so `to_crs` changes nothing here; it stays in as a safety step in case the city ever changes the projection. `COORDINATE_PRECISION=6` keeps six decimals, about 10 cm. **Output:** `../data/ruelles_vertes.geojson`

In [3]:
keep = keep.to_crs("EPSG:4326")
out_path = "../data/ruelles_vertes.geojson"
keep.to_file(out_path, driver="GeoJSON", COORDINATE_PRECISION=6)
print("Wrote", out_path, "-", round(os.path.getsize(out_path) / 1_000_000, 2), "MB",
      "(input was", round(os.path.getsize(in_path) / 1_000_000, 2), "MB)")

Wrote ../data/ruelles_vertes.geojson - 0.7 MB (input was 1.29 MB)


## 4) Check the file
The check cell from Lab 4, with the four settings changed for this file. It reads the file back the way the browser will.

In [4]:
# ---- settings: change these for your own data ----
FILE = "../data/ruelles_vertes.geojson"
WEST, EAST, SOUTH, NORTH = -73.98, -73.47, 45.40, 45.71   # a box around the island of Montréal, in degrees
EXPECTED = 1262           # how many features you expect (rows after your filters)
POPUP_FIELDS = ["borough", "year", "circulation", "greening"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Polygon']
All checks passed


## 5) Next: open the map
1. In VS Code, right-click `site/index.html` and choose Open with Live Server.
2. `site/app.js` fetches `../data/ruelles_vertes.geojson` and draws the alleys in green under the cat pins. Use the layer switcher (top right) to turn either layer on or off.